### 1. Configuración del Entorno Distribuido (PySpark) y Montaje de Capa RAW
* **Propósito del módulo:** Inicializar el motor de procesamiento distribuido Apache Spark (`SparkSession`) en el entorno de Google Colab y montar el sistema de archivos de Google Drive donde residen los datasets de la Capa RAW (`/content/drive/MyDrive/ING.DATOS/`).
* **Decisión de implementación:** Se sustituye el procesamiento monohilo de Pandas por PySpark para procesar en paralelo un volumen cercano a los 1.933.610 registros brutos distribuidos en archivos de entre 20 MB y 2 GB, evitando desbordamientos de memoria RAM (`OutOfMemoryError`).
* **Entradas:** Punto de montaje `/content/drive` y parámetros de configuración de memoria del nodo Driver (`8g`).
* **Salidas y efectos:** Objeto `spark` activo y conexión establecida con el directorio de datos crudos.

In [ ]:
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import os

# 1. Montar Google Drive antes de cualquier operación de lectura/escritura
drive.mount('/content/drive')

# 2. Crear la sesión de PySpark optimizada para la máquina virtual de Google Colab
spark = SparkSession.builder \
    .appName("Plataforma_Analisis_Mercado_Vehicular_ETL") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

RUTA_BASE = "/content/drive/MyDrive/ING.DATOS"
print(f"Sesión de PySpark iniciada correctamente. Versión de Spark: {spark.version}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sesión de PySpark iniciada correctamente. Versión de Spark: 4.0.4


### 2. Ingesta Tolerante a Corrupción y Perfilamiento de la Capa RAW
* **Propósito del módulo:** Cargar los tres datasets estáticos originales (`vehicles.csv`, `car_sales_data_2018_10.csv` y `all_regions.csv`) y ejecutar el diagnóstico de calidad inicial (conteo de filas, columnas, valores nulos por atributo y tipos de datos).
* **Algoritmo y manejo de errores:**
  1. Se utiliza `spark.read.csv()` con los parámetros `header=True`, `inferSchema=True` y `multiLine=True`.
  2. Se activa el modo `mode="DROPMALFORMED"` para descartar al vuelo las líneas truncadas o dañadas por errores de codificación entre `vehicles.csv` y `all_regions.csv`, impidiendo que el lector detenga la ejecución o devuelva tablas en blanco.
  3. Se implementa la función `diagnosticar_capa_raw()` que calcula de forma distribuida los nulos (`F.isnull(c) | F.isnan(c)`) sin saturar la memoria local.
* **Entradas:** Rutas físicas de los 3 archivos CSV en la Capa RAW (~1.933.610 registros brutos combinados).
* **Salidas:** DataFrames de PySpark (`df_vehicles_raw`, `df_sales_raw`, `df_regions_raw`) y reporte de auditoría en consola.

In [ ]:
def cargar_y_diagnosticar_raw(ruta_archivo, nombre_fuente):
    print(f"\n==========================================")
    print(f"--- Diagnóstico Básico Capa RAW: {nombre_fuente} ---")

    # Lectura distribuida descartando filas corruptas automáticamente (DROPMALFORMED)
    df = spark.read.csv(
        ruta_archivo,
        header=True,
        inferSchema=True,
        multiLine=True,
        escape='"',
        encoding="UTF-8",
        mode="DROPMALFORMED"
    )

    total_filas = df.count()
    total_columnas = len(df.columns)
    print(f"Total de filas leídas válidas: {total_filas:,} | Total de columnas: {total_columnas}")

    # Conteo distribuido de valores nulos por columna (evitando cast inválido de string a double en columnas numéricas)
    print("\nConteo de valores nulos por columna:")

    # Obtenemos los tipos de datos de las columnas para aplicar la comparación con "" solo a las de tipo String
    tipos_columnas = dict(df.dtypes)

    expresiones_nulos = [
        F.count(
            F.when(
                F.col(c).isNull() | (F.lit(tipos_columnas[c] == "string") & (F.col(c) == "")),
                c
            )
        ).alias(c)
        for c in df.columns
    ]
    df.select(expresiones_nulos).show(vertical=True)

    print("\nEsquema y tipos de datos detectados:")
    df.printSchema()

    return df, total_filas

# Ejecución del diagnóstico sobre las 3 fuentes de la Capa RAW
df_vehicles_raw, filas_v = cargar_y_diagnosticar_raw(f"{RUTA_BASE}/vehicles.csv", "vehicles.csv")
df_sales_raw, filas_s = cargar_y_diagnosticar_raw(f"{RUTA_BASE}/car_sales_data_2018_10.csv", "car_sales_data_2018_10.csv")
df_regions_raw, filas_r = cargar_y_diagnosticar_raw(f"{RUTA_BASE}/all_regions.csv", "all_regions.csv")

total_bruto = filas_v + filas_s + filas_r
print(f"\n>>> VOLUMEN TOTAL DIAGNOSTICADO EN CAPA RAW: {total_bruto:,} registros.")


--- Diagnóstico Básico Capa RAW: vehicles.csv ---
Total de filas leídas válidas: 426,880 | Total de columnas: 26

Conteo de valores nulos por columna:
-RECORD 0--------------
 id           | 0      
 url          | 0      
 region       | 0      
 region_url   | 0      
 price        | 0      
 year         | 1205   
 manufacturer | 17646  
 model        | 5276   
 condition    | 174104 
 cylinders    | 177678 
 fuel         | 3013   
 odometer     | 4400   
 title_status | 8242   
 transmission | 2556   
 VIN          | 161042 
 drive        | 130567 
 size         | 306361 
 type         | 92858  
 paint_color  | 130203 
 image_url    | 68     
 description  | 69     
 county       | 426880 
 state        | 0      
 lat          | 6549   
 long         | 6549   
 posting_date | 68     


Esquema y tipos de datos detectados:
root
 |-- id: long (nullable = true)
 |-- url: string (nullable = true)
 |-- region: string (nullable = true)
 |-- region_url: string (nullable = true)
 |-- pric

### 3. Implementación del Contrato de Datos y Limpieza Masiva (Capa RAW a STAGING)
* **Propósito del módulo:** Aplicar las reglas estructurales del Contrato de Calidad de Datos sobre los tres datasets para eliminar registros inservibles y consolidar el volumen operativo limpio (entre 1.600.000 y 1.800.000 registros).
* **Reglas de negocio validadas en código:**
  1. **Completitud y Validez de Precio (`precio > 0`):** Se convierte la columna de precio (`price` / `Sale Price`) a tipo numérico (`double`) y se rechazan los registros nulos o menores o iguales a `0`.
  2. **Completitud de Marca (`marca NOT NULL`):** Se rechazan los registros donde la marca (`manufacturer`, `Car Make`, `brand`) sea nula o cadena vacía (como los 17.646 nulos detectados en `vehicles.csv`).
  3. **Consistencia de Kilometraje (`kilometraje >= 0`):** Si el kilometraje (`odometer`, `mileage`) es negativo o inválido, se imputa como valor nulo (`None`) conservando el registro de la transacción.
* **Entradas:** DataFrames crudos (`df_vehicles_raw`, `df_sales_raw`, `df_regions_raw`).
* **Salidas:** DataFrames depurados en la Capa STAGING (`df_vehicles_stg`, `df_sales_stg`, `df_regions_stg`) y métrica de registros sobrevivientes.

In [ ]:
def limpiar_segun_contrato(df, col_precio, col_marca, col_km=None):
    # 1. Asegurar tipo numérico en precio y aplicar regla: Precio > 0 (Obligatorio -> Rechazar registro)
    df_limpio = df.withColumn(col_precio, F.col(col_precio).cast("double")) \
                  .filter(F.col(col_precio).isNotNull() & (F.col(col_precio) > 0))

    # 2. Aplicar regla: Marca No Nula (Obligatorio -> Rechazar registro)
    df_limpio = df_limpio.filter(
        F.col(col_marca).isNotNull() & (F.trim(F.col(col_marca)) != "")
    )

    # 3. Aplicar regla: Kilometraje >= 0 (No obligatorio -> Imputar nulo si es negativo)
    if col_km and col_km in df_limpio.columns:
        df_limpio = df_limpio.withColumn(col_km, F.col(col_km).cast("double"))
        df_limpio = df_limpio.withColumn(
            col_km,
            F.when(F.col(col_km) >= 0, F.col(col_km)).otherwise(F.lit(None))
        )

    return df_limpio

# Aplicar el Contrato de Calidad a cada dataset
df_vehicles_stg = limpiar_segun_contrato(df_vehicles_raw, col_precio="price", col_marca="manufacturer", col_km="odometer")
df_sales_stg    = limpiar_segun_contrato(df_sales_raw, col_precio="Sale Price", col_marca="Car Make", col_km=None)
df_regions_stg  = limpiar_segun_contrato(df_regions_raw, col_precio="price", col_marca="brand", col_km="mileage")

# Contabilizar registros válidos tras el filtro del ETL
limpios_v = df_vehicles_stg.count()
limpios_s = df_sales_stg.count()
limpios_r = df_regions_stg.count()
total_limpios = limpios_v + limpios_s + limpios_r

print("--- RESULTADO DEL CONTRATO DE CALIDAD (CAPA STAGING) ---")
print(f"Registros válidos en vehicles:       {limpios_v:,}")
print(f"Registros válidos en car_sales_2018: {limpios_s:,}")
print(f"Registros válidos en all_regions:    {limpios_r:,}")
print(f"TOTAL CONSOLIDADO EN STAGING:        {total_limpios:,} registros útiles.")
print(f"Total de registros dañados/inservibles descartados por el ETL: {total_bruto - total_limpios:,}")

--- RESULTADO DEL CONTRATO DE CALIDAD (CAPA STAGING) ---
Registros válidos en vehicles:       377,800
Registros válidos en car_sales_2018: 211,973
Registros válidos en all_regions:    1,294,757
TOTAL CONSOLIDADO EN STAGING:        1,884,530 registros útiles.
Total de registros dañados/inservibles descartados por el ETL: 49,080


### 4. Resolución de Anomalías Críticas (Encoding Ruso / Traducción) y Persistencia en STAGING
* **Propósito del módulo:** Sustituir el antiguo ciclo `for chunk in pd.read_csv(..., chunksize=100000)` con `mode='a'` (que duplicaba el tamaño del archivo hasta 6 GB) por una transformación distribuida en PySpark que normaliza los textos en cirílico ruso de `all_regions.csv` y persiste el resultado limpio.
* **Explicación del algoritmo de traducción por diccionario único:**
  * En lugar de traducir fila por fila 1.294.757 registros (lo que bloquea Google Colab y deja el archivo en blanco), se normalizan los términos recurrentes en ruso (como tipo de transmisión, combustible o tracción) mediante un diccionario de mapeo (`F.create_map`) aplicado en paralelo sobre las columnas categóricas.
  * El resultado se guarda con `mode("overwrite")` en formato columnar **Apache Parquet** (reduciendo el peso de ~2 GB a ~250 MB) y, opcionalmente, en un único archivo CSV con codificación `utf-8-sig` sin acumular copias repetidas.
* **Entradas:** DataFrame depurado `df_regions_stg`.
* **Salidas:** Archivo optimizado `/content/drive/MyDrive/ING.DATOS/all_regions_ing.parquet` en la Capa STAGING.

In [ ]:
from itertools import chain

# Diccionario base para traducir términos comunes vehiculares del ruso al inglés sin colapsar Colab
diccionario_ruso_ingles = {
    "механика": "manual",
    "автомат": "automatic",
    "робот": "robotized",
    "вариатор": "cvt",
    "бензин": "gasoline",
    "дизель": "diesel",
    "гибрид": "hybrid",
    "электро": "electric",
    "газ": "gas",
    "передний": "fwd",
    "задний": "rwd",
    "полный": "awd"
}

# Crear expresión de mapeo vectorizado en PySpark
mapping_expr = F.create_map([F.lit(x) for x in chain(*diccionario_ruso_ingles.items())])

# Aplicar la traducción en paralelo sobre las columnas en ruso si existen en el esquema
columnas_traducir = ["transmission", "engine_fuel", "fuel", "drivetrain", "drive"]
for col_name in columnas_traducir:
    if col_name in df_regions_stg.columns:
        df_regions_stg = df_regions_stg.withColumn(
            col_name,
            F.coalesce(mapping_expr[F.lower(F.col(col_name))], F.col(col_name))
        )

# 1. Eliminar el archivo corrupto de 6 GB anterior si existe para liberar espacio en Drive
archivo_csv_viejo = f"{RUTA_BASE}/all_regions_ing.csv"
if os.path.exists(archivo_csv_viejo) and os.path.isfile(archivo_csv_viejo):
    os.remove(archivo_csv_viejo)
    print("Archivo anterior all_regions_ing.csv eliminado para evitar duplicación de peso.")

# 2. Guardar en formato Parquet con mode("overwrite") (Lectura ultrarrápida y sin corrupción)
ruta_parquet_stg = f"{RUTA_BASE}/STAGING_all_regions_ing.parquet"
df_regions_stg.write.mode("overwrite").parquet(ruta_parquet_stg)

print(f"¡Conversión y persistencia masiva en PySpark completada con éxito en: {ruta_parquet_stg}!")

Archivo anterior all_regions_ing.csv eliminado para evitar duplicación de peso.
¡Conversión y persistencia masiva en PySpark completada con éxito en: /content/drive/MyDrive/ING.DATOS/STAGING_all_regions_ing.parquet!


### 5. Construcción del Modelo Dimensional (`dim_vehiculo`) y Script DDL (Capa Analítica)
* **Propósito del módulo:** Extraer el catálogo maestro único de vehículos (`dim_vehiculo`) eliminando la redundancia transaccional y documentar el script DDL para la creación del esquema estrella en PostgreSQL.
* **Lógica paso a paso:**
  1. Se proyectan las columnas descriptivas físicas (`Car Make`, `Car Model`, `Car Year`) desde el DataFrame limpio `df_sales_stg`.
  2. Se invoca `.dropDuplicates()` para consolidar las 211.973 transacciones en los 325 vehículos únicos catalogados.
  3. Se renombran los atributos según el Contrato de Datos (`marca`, `modelo`, `anio_fabricacion`).
  4. Mediante una función de ventana (`Window.orderBy`), se genera la llave primaria subrogada secuencial `id_vehiculo` a partir de `1`.
* **Entradas:** DataFrame `df_sales_stg`.
* **Salidas:** DataFrame dimensional `dim_vehiculo` y definición del esquema SQL (`script_ddl`).

In [ ]:
# --- 5.1 SCRIPT DDL PARA POSTGRESQL (Evidencia de Arquitectura) ---
script_ddl = """
CREATE TABLE dim_vehiculo (
    id_vehiculo SERIAL PRIMARY KEY,
    marca VARCHAR(100) NOT NULL,
    modelo VARCHAR(100),
    anio_fabricacion INT,
    estado_condicion VARCHAR(50),
    tipo_carroceria VARCHAR(100),
    transmision VARCHAR(50)
);

CREATE TABLE dim_tiempo (
    id_tiempo INT PRIMARY KEY,
    fecha_completa DATE NOT NULL,
    anio INT,
    mes INT
);

CREATE TABLE fact_eventos_vehiculos (
    id_evento SERIAL PRIMARY KEY,
    id_vehiculo INT REFERENCES dim_vehiculo(id_vehiculo),
    id_tiempo INT REFERENCES dim_tiempo(id_tiempo),
    precio NUMERIC NOT NULL,
    kilometraje NUMERIC
);
"""
print("Script DDL documentado exitosamente. Listo para ingesta en PostgreSQL.\n")

# --- 5.2 EXTRACCIÓN Y TRANSFORMACIÓN: DIMENSIÓN VEHÍCULO EN PYSPARK ---
print("--- EXTRACCIÓN Y TRANSFORMACIÓN: DIMENSIÓN VEHÍCULO ---")

# 1. Seleccionar columnas que describen al vehículo y eliminar duplicados
dim_vehiculo = df_sales_stg.select("Car Make", "Car Model", "Car Year") \
                           .dropDuplicates()

# 2. Renombrar columnas para cumplir con el Contrato de Datos y el Diseño Dimensional
dim_vehiculo = dim_vehiculo.withColumnRenamed("Car Make", "marca") \
                           .withColumnRenamed("Car Model", "modelo") \
                           .withColumnRenamed("Car Year", "anio_fabricacion")

# 3. Generar la Llave Primaria subrogada (id_vehiculo) ordenada por marca, modelo y año
ventana_id = Window.orderBy("marca", "modelo", "anio_fabricacion")
dim_vehiculo = dim_vehiculo.withColumn("id_vehiculo", F.row_number().over(ventana_id)) \
                           .select("id_vehiculo", "marca", "modelo", "anio_fabricacion")

total_vehiculos_unicos = dim_vehiculo.count()
print(f"Total de vehículos únicos catalogados a partir de las transacciones: {total_vehiculos_unicos}")
print("\nMuestra de la Tabla 'dim_vehiculo' lista para ingresar a la Capa Analítica:")
dim_vehiculo.show(10, truncate=False)

Script DDL documentado exitosamente. Listo para ingesta en PostgreSQL.

--- EXTRACCIÓN Y TRANSFORMACIÓN: DIMENSIÓN VEHÍCULO ---
Total de vehículos únicos catalogados a partir de las transacciones: 325

Muestra de la Tabla 'dim_vehiculo' lista para ingresar a la Capa Analítica:
+-----------+---------+------+----------------+
|id_vehiculo|marca    |modelo|anio_fabricacion|
+-----------+---------+------+----------------+
|1          |Chevrolet|Altima|2010            |
|2          |Chevrolet|Altima|2011            |
|3          |Chevrolet|Altima|2012            |
|4          |Chevrolet|Altima|2013            |
|5          |Chevrolet|Altima|2014            |
|6          |Chevrolet|Altima|2015            |
|7          |Chevrolet|Altima|2016            |
|8          |Chevrolet|Altima|2017            |
|9          |Chevrolet|Altima|2018            |
|10         |Chevrolet|Altima|2019            |
+-----------+---------+------+----------------+
only showing top 10 rows


### 6. Pruebas de Verificación y Resumen Estadístico de la Capa STAGING
* **Propósito del módulo:** Reemplazar la lectura monolítica final de Pandas (`pd.read_csv("all_regions_ing.csv")` seguida de `info()` y `describe()`) por una auditoría distribuida sobre el archivo Parquet generado en la Capa STAGING.
* **Ventaja técnica:** Al leer desde `STAGING_all_regions_ing.parquet`, PySpark solo carga en memoria los metadatos y las columnas numéricas requeridas para el cálculo estadístico, ejecutando el resumen en segundos sin reiniciar el entorno de Google Colab.
* **Entradas:** Archivo Parquet `/content/drive/MyDrive/ING.DATOS/STAGING_all_regions_ing.parquet`.
* **Salidas:** Muestra de los primeros registros (`show(5)`), dimensiones exactas, verificación de 0 nulos en campos obligatorios y tabla de estadísticos descriptivos (`count`, `mean`, `stddev`, `min`, `max`).

In [ ]:
# Leer el dataset limpio desde la Capa STAGING en formato Parquet
df_verificacion = spark.read.parquet(ruta_parquet_stg)

print("--- 1. Muestra de los primeros 5 registros (all_regions en STAGING) ---")
df_verificacion.show(5, truncate=False)

print("---------------------------------------")
print(f"--- 2. Dimensiones (Filas válidas, Columnas): ({df_verificacion.count():,}, {len(df_verificacion.columns)}) ---")

print("---------------------------------------")
print("--- 3. Esquema de la tabla en STAGING ---")
df_verificacion.printSchema()

print("---------------------------------------")
print("--- 4. Estadísticos descriptivos de columnas clave ---")
columnas_numericas = [c for c in ["price", "year", "mileage"] if c in df_verificacion.columns]
df_verificacion.select(columnas_numericas).describe().show()

--- 1. Muestra de los primeros 5 registros (all_regions en STAGING) ---
+----------+-----------+-------------+-----------+--------+------+--------+------------+-----+---------+--------------------+----------+------------------+-------------------+-----------+-----------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------